This notebook is for quality control of the partially extracted Wikidata location build.

1. validate the Wikidata tables and internal consistency;
2. inspect extraction coverage, language coverage, location-kind coverage, coordinates, P31 mapping, and lookup ambiguity;
3. estimate truncation/corruption bias using QID distribution and linked-record signals;

In [ ]:
import re
import sqlite3
import textwrap
import unicodedata
from pathlib import Path
from typing import Sequence

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd().parent

WIKIDATA_DB_PATH = PROJECT_ROOT / "data" / "names.sqlite"
GEONAMES_DB_PATH = WIKIDATA_DB_PATH

SAMPLE_LIMIT = 50
TOP_LIMIT = 50

IMPORTANT_GEO_LANGS = [
    "en", "fr", "de", "es", "it", "pt", "nl", "ru", "uk", "pl", "cs", "el", "tr", "ar", "fa", "he",
    "zh", "ja", "ko", "la", "grc", "ang", "non", "got", "cu", "hy", "ka", "kk", "tt", "mns", "kca",
]

print("PROJECT_ROOT     =", PROJECT_ROOT)
print("WIKIDATA_DB_PATH =", WIKIDATA_DB_PATH, "exists=", WIKIDATA_DB_PATH.exists())
print("GEONAMES_DB_PATH =", GEONAMES_DB_PATH, "exists=", GEONAMES_DB_PATH.exists())


## 1. Database helpers

These helpers keep queries explicit while supporting either a single merged DB or a separate attached GeoNames DB.


In [ ]:

def _same_path(a: Path, b: Path) -> bool:
    try:
        return a.resolve() == b.resolve()
    except FileNotFoundError:
        return a == b


GEO_SCHEMA = "main" if _same_path(WIKIDATA_DB_PATH, GEONAMES_DB_PATH) else "geo"


def get_conn(*, attach_geonames: bool = False) -> sqlite3.Connection:
    conn = sqlite3.connect(WIKIDATA_DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON;")
    conn.execute("PRAGMA temp_store = MEMORY;")
    conn.execute("PRAGMA cache_size = -200000;")
    if attach_geonames and GEO_SCHEMA == "geo":
        conn.execute("ATTACH DATABASE ? AS geo;", (str(GEONAMES_DB_PATH),))
    return conn


def q(sql: str, params: Sequence | None = None, *, attach_geonames: bool = False) -> pd.DataFrame:
    sql = textwrap.dedent(sql).strip()
    with get_conn(attach_geonames=attach_geonames) as conn:
        return pd.read_sql_query(sql, conn, params=params or ())


def scalar(sql: str, params: Sequence | None = None, *, attach_geonames: bool = False):
    df = q(sql, params=params, attach_geonames=attach_geonames)
    if df.empty:
        return None
    return df.iat[0, 0]


def show(title: str, df: pd.DataFrame, *, max_rows: int | None = None) -> pd.DataFrame:
    display(Markdown(f"### {title}"))
    if max_rows is not None:
        display(df.head(max_rows))
    else:
        display(df)
    return df


def table_exists(name: str, *, schema: str = "main", attach_geonames: bool = False) -> bool:
    return bool(scalar(
        f"SELECT 1 FROM {schema}.sqlite_master WHERE type = 'table' AND name = ? LIMIT 1",
        (name,),
        attach_geonames=attach_geonames,
    ))


def index_exists(name: str, *, schema: str = "main", attach_geonames: bool = False) -> bool:
    return bool(scalar(
        f"SELECT 1 FROM {schema}.sqlite_master WHERE type = 'index' AND name = ? LIMIT 1",
        (name,),
        attach_geonames=attach_geonames,
    ))


def pct(num, den) -> float:
    if den in (0, None) or pd.isna(den):
        return float("nan")
    return 100.0 * num / den


def add_pct(df: pd.DataFrame, numerator: str, denominator: str, out_col: str = "pct") -> pd.DataFrame:
    df = df.copy()
    df[out_col] = [pct(n, d) for n, d in zip(df[numerator], df[denominator])]
    return df


def sql_in(values: Sequence[str]) -> str:
    return ", ".join("?" for _ in values)


def normalize_name_for_qc(s: str | None) -> str | None:
    """Approximation of the conservative lookup normalization: NFC + whitespace collapse + casefold."""
    if s is None:
        return None
    s = unicodedata.normalize("NFC", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s.casefold()


def has_control_chars(s: str | None) -> int:
    if not s:
        return 0
    return int(any(unicodedata.category(ch).startswith("C") for ch in s))


def qid_num_expr(col: str = "qid") -> str:
    return f"CASE WHEN {col} GLOB 'Q[0-9]*' THEN CAST(SUBSTR({col}, 2) AS INTEGER) END"


def check_required_tables() -> pd.DataFrame:
    required = [
        "wikidata_location",
        "wikidata_location_geonames",
        "wikidata_location_p31",
        "wikidata_lang_norm",
        "wikidata_location_name",
    ]
    rows = []
    for t in required:
        rows.append({"table": t, "exists": table_exists(t)})
    return pd.DataFrame(rows)

show("Required Wikidata tables", check_required_tables())


## 2. Schema, counts, and integrity checks

This section checks that the extraction produced the expected tables and that there are no obvious orphaned records. It does **not** assume SQLite foreign keys were enforced during import.


In [ ]:

expected_wd_columns = {
    "wikidata_location": ["qid", "kind", "lat", "lon"],
    "wikidata_location_geonames": ["qid", "geonames_id"],
    "wikidata_location_p31": ["qid", "p31_qid"],
    "wikidata_lang_norm": ["wd_lang", "geo_lang", "iso639_3", "iso639_1"],
    "wikidata_location_name": ["qid", "wd_lang", "geo_lang", "name", "name_norm", "term_type"],
}

schema_rows = []
for table, expected_cols in expected_wd_columns.items():
    if not table_exists(table):
        schema_rows.append({"table": table, "status": "MISSING", "missing_columns": ", ".join(expected_cols), "extra_columns": ""})
        continue
    cols = q(f"PRAGMA table_info({table})")
    actual = cols["name"].tolist()
    schema_rows.append({
        "table": table,
        "status": "ok" if set(expected_cols).issubset(actual) else "column mismatch",
        "missing_columns": ", ".join([c for c in expected_cols if c not in actual]),
        "extra_columns": ", ".join([c for c in actual if c not in expected_cols]),
        "actual_columns": ", ".join(actual),
    })

show("Wikidata schema check", pd.DataFrame(schema_rows))

count_sql = """
SELECT 'wikidata_location' AS table_name, COUNT(*) AS rows FROM wikidata_location
UNION ALL SELECT 'wikidata_location_name', COUNT(*) FROM wikidata_location_name
UNION ALL SELECT 'wikidata_location_p31', COUNT(*) FROM wikidata_location_p31
UNION ALL SELECT 'wikidata_location_geonames', COUNT(*) FROM wikidata_location_geonames
UNION ALL SELECT 'wikidata_lang_norm', COUNT(*) FROM wikidata_lang_norm
"""
wd_counts = q(count_sql)
show("Wikidata table row counts", wd_counts)

with get_conn() as conn:
    integrity_check = pd.DataFrame(conn.execute("PRAGMA integrity_check;").fetchall(), columns=["integrity_check"])
    foreign_key_check = pd.DataFrame(conn.execute("PRAGMA foreign_key_check;").fetchall())

show("SQLite integrity_check", integrity_check)
show("SQLite foreign_key_check", foreign_key_check if not foreign_key_check.empty else pd.DataFrame([{"result": "no FK violations reported"}]))


In [ ]:

internal_consistency = q("""
WITH metrics AS (
    SELECT 'names with missing parent qid' AS check_name, COUNT(*) AS n
    FROM wikidata_location_name n
    LEFT JOIN wikidata_location l ON l.qid = n.qid
    WHERE l.qid IS NULL

    UNION ALL
    SELECT 'p31 rows with missing parent qid', COUNT(*)
    FROM wikidata_location_p31 p
    LEFT JOIN wikidata_location l ON l.qid = p.qid
    WHERE l.qid IS NULL

    UNION ALL
    SELECT 'GeoNames link rows with missing parent qid', COUNT(*)
    FROM wikidata_location_geonames g
    LEFT JOIN wikidata_location l ON l.qid = g.qid
    WHERE l.qid IS NULL

    UNION ALL
    SELECT 'name rows with wd_lang missing from wikidata_lang_norm', COUNT(*)
    FROM wikidata_location_name n
    LEFT JOIN wikidata_lang_norm ln ON ln.wd_lang = n.wd_lang
    WHERE ln.wd_lang IS NULL

    UNION ALL
    SELECT 'name rows where stored geo_lang disagrees with wikidata_lang_norm', COUNT(*)
    FROM wikidata_location_name n
    JOIN wikidata_lang_norm ln ON ln.wd_lang = n.wd_lang
    WHERE COALESCE(n.geo_lang, '') <> COALESCE(ln.geo_lang, '')

    UNION ALL
    SELECT 'locations with no names', COUNT(*)
    FROM wikidata_location l
    LEFT JOIN wikidata_location_name n ON n.qid = l.qid
    WHERE n.qid IS NULL

    UNION ALL
    SELECT 'locations with no P31 row', COUNT(*)
    FROM wikidata_location l
    LEFT JOIN wikidata_location_p31 p ON p.qid = l.qid
    WHERE p.qid IS NULL
)
SELECT * FROM metrics ORDER BY n DESC;
""")
show("Internal consistency checks", internal_consistency)


## 3. Wikidata extraction overview

These are the high-level survival metrics for the corrupted dump build: how many locations survived, how many have coordinates, names, language mappings, P31 values, and GeoNames links.


In [ ]:

overview = q("""
WITH
name_counts AS (
    SELECT
        qid,
        COUNT(*) AS name_rows,
        COUNT(DISTINCT wd_lang) AS wd_langs,
        COUNT(DISTINCT geo_lang) AS geo_langs,
        SUM(CASE WHEN geo_lang IS NULL THEN 1 ELSE 0 END) AS unmapped_name_rows
    FROM wikidata_location_name
    GROUP BY qid
),
p31_counts AS (
    SELECT qid, COUNT(DISTINCT p31_qid) AS p31_count
    FROM wikidata_location_p31
    GROUP BY qid
),
geonames_counts AS (
    SELECT qid, COUNT(DISTINCT geonames_id) AS geonames_ids
    FROM wikidata_location_geonames
    GROUP BY qid
)
SELECT
    COUNT(*) AS locations,
    SUM(CASE WHEN l.lat IS NOT NULL AND l.lon IS NOT NULL THEN 1 ELSE 0 END) AS with_coords,
    SUM(CASE WHEN COALESCE(n.name_rows, 0) > 0 THEN 1 ELSE 0 END) AS with_names,
    SUM(CASE WHEN COALESCE(n.geo_langs, 0) > 0 THEN 1 ELSE 0 END) AS with_mapped_geo_lang,
    SUM(CASE WHEN COALESCE(p.p31_count, 0) > 0 THEN 1 ELSE 0 END) AS with_p31,
    SUM(CASE WHEN COALESCE(g.geonames_ids, 0) > 0 THEN 1 ELSE 0 END) AS with_geonames_link,
    SUM(COALESCE(n.name_rows, 0)) AS total_name_rows,
    SUM(COALESCE(n.unmapped_name_rows, 0)) AS total_unmapped_name_rows,
    AVG(COALESCE(n.name_rows, 0)) AS avg_names_per_location,
    AVG(COALESCE(n.geo_langs, 0)) AS avg_geo_langs_per_location
FROM wikidata_location l
LEFT JOIN name_counts n ON n.qid = l.qid
LEFT JOIN p31_counts p ON p.qid = l.qid
LEFT JOIN geonames_counts g ON g.qid = l.qid;
""")
for col in ["with_coords", "with_names", "with_mapped_geo_lang", "with_p31", "with_geonames_link"]:
    overview[col + "_pct"] = overview[col] / overview["locations"] * 100
if int(overview.loc[0, "total_name_rows"] or 0):
    overview["unmapped_name_rows_pct"] = overview["total_unmapped_name_rows"] / overview["total_name_rows"] * 100
show("Wikidata extraction overview", overview.T.reset_index().rename(columns={"index": "metric", 0: "value"}))


In [ ]:

kind_overview = q("""
WITH
name_counts AS (
    SELECT
        qid,
        COUNT(*) AS name_rows,
        COUNT(DISTINCT wd_lang) AS wd_langs,
        COUNT(DISTINCT geo_lang) AS geo_langs,
        SUM(CASE WHEN geo_lang IS NULL THEN 1 ELSE 0 END) AS unmapped_name_rows
    FROM wikidata_location_name
    GROUP BY qid
),
p31_counts AS (
    SELECT qid, COUNT(DISTINCT p31_qid) AS p31_count
    FROM wikidata_location_p31
    GROUP BY qid
),
geonames_counts AS (
    SELECT qid, COUNT(DISTINCT geonames_id) AS geonames_ids
    FROM wikidata_location_geonames
    GROUP BY qid
)
SELECT
    l.kind,
    COUNT(*) AS locations,
    SUM(CASE WHEN l.lat IS NOT NULL AND l.lon IS NOT NULL THEN 1 ELSE 0 END) AS with_coords,
    SUM(CASE WHEN COALESCE(n.name_rows, 0) > 0 THEN 1 ELSE 0 END) AS with_names,
    SUM(CASE WHEN COALESCE(n.geo_langs, 0) > 0 THEN 1 ELSE 0 END) AS with_mapped_geo_lang,
    SUM(CASE WHEN COALESCE(g.geonames_ids, 0) > 0 THEN 1 ELSE 0 END) AS with_geonames_link,
    SUM(COALESCE(n.name_rows, 0)) AS name_rows,
    AVG(COALESCE(n.name_rows, 0)) AS avg_names,
    AVG(COALESCE(n.geo_langs, 0)) AS avg_geo_langs,
    AVG(COALESCE(p.p31_count, 0)) AS avg_p31
FROM wikidata_location l
LEFT JOIN name_counts n ON n.qid = l.qid
LEFT JOIN p31_counts p ON p.qid = l.qid
LEFT JOIN geonames_counts g ON g.qid = l.qid
GROUP BY l.kind
ORDER BY locations DESC;
""")
for col in ["with_coords", "with_names", "with_mapped_geo_lang", "with_geonames_link"]:
    kind_overview[col + "_pct"] = kind_overview[col] / kind_overview["locations"] * 100
show("Coverage by Wikidata-derived NTE kind", kind_overview)


In [ ]:

# Visual overview: location count by kind, limited to top categories.
plot_df = kind_overview.head(30).sort_values("locations")
ax = plot_df.plot.barh(x="kind", y="locations", legend=False, figsize=(10, 8))
ax.set_title("Wikidata locations by kind")
ax.set_xlabel("locations")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 4. Entity-level name and language coverage

The central question for NTE is not just “how many places survived,” but whether each surviving place has enough language labels to be useful as lookup data.


In [ ]:

name_bucket_distribution = q("""
WITH name_counts AS (
    SELECT l.qid, l.kind, COUNT(n.name) AS name_rows, COUNT(DISTINCT n.geo_lang) AS geo_langs
    FROM wikidata_location l
    LEFT JOIN wikidata_location_name n ON n.qid = l.qid
    GROUP BY l.qid, l.kind
), bucketed AS (
    SELECT
        kind,
        CASE
            WHEN name_rows = 0 THEN '00 no names'
            WHEN name_rows = 1 THEN '01 one name'
            WHEN name_rows BETWEEN 2 AND 3 THEN '02 2-3 names'
            WHEN name_rows BETWEEN 4 AND 5 THEN '03 4-5 names'
            WHEN name_rows BETWEEN 6 AND 10 THEN '04 6-10 names'
            WHEN name_rows BETWEEN 11 AND 25 THEN '05 11-25 names'
            WHEN name_rows BETWEEN 26 AND 50 THEN '06 26-50 names'
            ELSE '07 >50 names'
        END AS name_bucket,
        COUNT(*) AS locations
    FROM name_counts
    GROUP BY kind, name_bucket
)
SELECT * FROM bucketed ORDER BY kind, name_bucket;
""")
show("Name-count buckets by kind", name_bucket_distribution)


In [ ]:

lang_bucket_distribution = q("""
WITH lang_counts AS (
    SELECT l.qid, l.kind, COUNT(DISTINCT n.geo_lang) AS geo_langs
    FROM wikidata_location l
    LEFT JOIN wikidata_location_name n ON n.qid = l.qid AND n.geo_lang IS NOT NULL
    GROUP BY l.qid, l.kind
), bucketed AS (
    SELECT
        kind,
        CASE
            WHEN geo_langs = 0 THEN '00 no mapped languages'
            WHEN geo_langs = 1 THEN '01 one language'
            WHEN geo_langs BETWEEN 2 AND 3 THEN '02 2-3 languages'
            WHEN geo_langs BETWEEN 4 AND 5 THEN '03 4-5 languages'
            WHEN geo_langs BETWEEN 6 AND 10 THEN '04 6-10 languages'
            WHEN geo_langs BETWEEN 11 AND 25 THEN '05 11-25 languages'
            WHEN geo_langs BETWEEN 26 AND 50 THEN '06 26-50 languages'
            ELSE '07 >50 languages'
        END AS geo_lang_bucket,
        COUNT(*) AS locations
    FROM lang_counts
    GROUP BY kind, geo_lang_bucket
)
SELECT * FROM bucketed ORDER BY kind, geo_lang_bucket;
""")
show("Mapped-language buckets by kind", lang_bucket_distribution)


In [ ]:

important_placeholders = sql_in(IMPORTANT_GEO_LANGS)
important_lang_coverage = q(f"""
WITH total AS (
    SELECT COUNT(*) AS locations FROM wikidata_location
), per_lang AS (
    SELECT
        n.geo_lang,
        COUNT(*) AS name_rows,
        COUNT(DISTINCT n.qid) AS locations_with_lang
    FROM wikidata_location_name n
    WHERE n.geo_lang IN ({important_placeholders})
    GROUP BY n.geo_lang
)
SELECT
    p.geo_lang,
    p.name_rows,
    p.locations_with_lang,
    ROUND(100.0 * p.locations_with_lang / total.locations, 3) AS locations_pct
FROM per_lang p CROSS JOIN total
ORDER BY locations_with_lang DESC;
""", IMPORTANT_GEO_LANGS)
show("Coverage for important language buckets", important_lang_coverage)


In [ ]:

important_by_kind_exprs = []
params = []
for lang in IMPORTANT_GEO_LANGS:
    safe_col = re.sub(r"[^0-9A-Za-z_]", "_", lang)
    important_by_kind_exprs.append(f"SUM(CASE WHEN f.has_{safe_col} = 1 THEN 1 ELSE 0 END) AS has_{safe_col}")

flag_exprs = []
for lang in IMPORTANT_GEO_LANGS:
    safe_col = re.sub(r"[^0-9A-Za-z_]", "_", lang)
    flag_exprs.append(f"MAX(CASE WHEN geo_lang = ? THEN 1 ELSE 0 END) AS has_{safe_col}")
    params.append(lang)

important_by_kind = q(f"""
WITH flags AS (
    SELECT qid, {", ".join(flag_exprs)}
    FROM wikidata_location_name
    GROUP BY qid
)
SELECT
    l.kind,
    COUNT(*) AS locations,
    {", ".join(important_by_kind_exprs)}
FROM wikidata_location l
LEFT JOIN flags f ON f.qid = l.qid
GROUP BY l.kind
ORDER BY locations DESC;
""", params)

# Add percentages for easier scanning.
for lang in IMPORTANT_GEO_LANGS:
    col = "has_" + re.sub(r"[^0-9A-Za-z_]", "_", lang)
    if col in important_by_kind.columns:
        important_by_kind[col + "_pct"] = important_by_kind[col] / important_by_kind["locations"] * 100

show("Important-language entity coverage by kind", important_by_kind)


In [ ]:

poorly_covered_samples = q(f"""
WITH lang_counts AS (
    SELECT l.qid, l.kind, COUNT(DISTINCT n.geo_lang) AS geo_langs, COUNT(n.name) AS name_rows
    FROM wikidata_location l
    LEFT JOIN wikidata_location_name n ON n.qid = l.qid AND n.geo_lang IS NOT NULL
    GROUP BY l.qid, l.kind
)
SELECT lc.*, l.lat, l.lon
FROM lang_counts lc
JOIN wikidata_location l ON l.qid = lc.qid
WHERE lc.geo_langs <= 1
ORDER BY lc.kind, lc.geo_langs, lc.name_rows
LIMIT {SAMPLE_LIMIT};
""")
show("Sample: locations with 0-1 mapped languages", poorly_covered_samples)


## 5. Language normalization quality


In [ ]:

lang_norm_summary = q("""
SELECT
    COUNT(*) AS wd_lang_rows,
    COUNT(geo_lang) AS rows_with_geo_lang,
    COUNT(DISTINCT geo_lang) AS distinct_geo_langs,
    COUNT(iso639_3) AS rows_with_iso639_3,
    COUNT(iso639_1) AS rows_with_iso639_1
FROM wikidata_lang_norm;
""")
show("wikidata_lang_norm summary", lang_norm_summary.T.reset_index().rename(columns={"index": "metric", 0: "value"}))

wd_lang_usage = q(f"""
SELECT
    n.wd_lang,
    n.geo_lang,
    COUNT(*) AS name_rows,
    COUNT(DISTINCT n.qid) AS locations,
    COUNT(DISTINCT n.name_norm) AS distinct_normalized_names
FROM wikidata_location_name n
GROUP BY n.wd_lang, n.geo_lang
ORDER BY name_rows DESC
LIMIT {TOP_LIMIT};
""")
show("Top Wikidata language tags in names", wd_lang_usage)


In [ ]:

unmapped_langs = q(f"""
SELECT
    n.wd_lang,
    COUNT(*) AS name_rows,
    COUNT(DISTINCT n.qid) AS locations,
    COUNT(DISTINCT n.name_norm) AS distinct_normalized_names,
    MIN(n.name) AS sample_name
FROM wikidata_location_name n
WHERE n.geo_lang IS NULL
GROUP BY n.wd_lang
ORDER BY name_rows DESC
LIMIT {TOP_LIMIT};
""")
show("Top wd_lang values with no geo_lang mapping", unmapped_langs)

geo_lang_collapses = q(f"""
SELECT
    geo_lang,
    COUNT(DISTINCT wd_lang) AS wd_lang_variants,
    GROUP_CONCAT(DISTINCT wd_lang) AS wd_langs,
    COUNT(*) AS norm_rows
FROM wikidata_lang_norm
WHERE geo_lang IS NOT NULL
GROUP BY geo_lang
HAVING COUNT(DISTINCT wd_lang) > 1
ORDER BY wd_lang_variants DESC, geo_lang
LIMIT {TOP_LIMIT};
""")
show("geo_lang buckets receiving multiple wd_lang variants", geo_lang_collapses)


In [ ]:

langs_of_interest = q(f"""
SELECT
    ln.wd_lang,
    ln.geo_lang,
    ln.iso639_3,
    ln.iso639_1,
    COUNT(n.name) AS name_rows,
    COUNT(DISTINCT n.qid) AS locations,
    MIN(n.name) AS sample_name
FROM wikidata_lang_norm ln
LEFT JOIN wikidata_location_name n ON n.wd_lang = ln.wd_lang
WHERE ln.wd_lang IN ({sql_in(IMPORTANT_GEO_LANGS)})
   OR ln.geo_lang IN ({sql_in(IMPORTANT_GEO_LANGS)})
GROUP BY ln.wd_lang, ln.geo_lang, ln.iso639_3, ln.iso639_1
ORDER BY COALESCE(name_rows, 0) DESC, ln.geo_lang, ln.wd_lang;
""", IMPORTANT_GEO_LANGS + IMPORTANT_GEO_LANGS)
show("Configured languages of interest: normalization and coverage", langs_of_interest)


## 6. Location kind and P31 mapping audit


In [ ]:

p31_top = q(f"""
SELECT
    p.p31_qid,
    l.kind,
    COUNT(*) AS locations
FROM wikidata_location_p31 p
JOIN wikidata_location l ON l.qid = p.qid
GROUP BY p.p31_qid, l.kind
ORDER BY locations DESC
LIMIT {TOP_LIMIT};
""")
show("Top P31 values by kind", p31_top)

p31_multi_kind = q(f"""
SELECT
    p.p31_qid,
    COUNT(DISTINCT l.kind) AS kind_count,
    GROUP_CONCAT(DISTINCT l.kind) AS kinds,
    COUNT(*) AS locations
FROM wikidata_location_p31 p
JOIN wikidata_location l ON l.qid = p.qid
GROUP BY p.p31_qid
HAVING COUNT(DISTINCT l.kind) > 1
ORDER BY kind_count DESC, locations DESC
LIMIT {TOP_LIMIT};
""")
show("P31 values mapped into multiple NTE kinds", p31_multi_kind)


In [ ]:

p31_count_distribution = q("""
WITH p31_counts AS (
    SELECT l.qid, l.kind, COUNT(DISTINCT p.p31_qid) AS p31_count
    FROM wikidata_location l
    LEFT JOIN wikidata_location_p31 p ON p.qid = l.qid
    GROUP BY l.qid, l.kind
)
SELECT
    kind,
    p31_count,
    COUNT(*) AS locations
FROM p31_counts
GROUP BY kind, p31_count
ORDER BY kind, p31_count;
""")
show("P31-count distribution by kind", p31_count_distribution)

multi_p31_samples = q(f"""
WITH p31_counts AS (
    SELECT qid, COUNT(DISTINCT p31_qid) AS p31_count
    FROM wikidata_location_p31
    GROUP BY qid
    HAVING COUNT(DISTINCT p31_qid) >= 4
)
SELECT l.qid, l.kind, pc.p31_count, GROUP_CONCAT(p.p31_qid) AS p31_qids
FROM p31_counts pc
JOIN wikidata_location l ON l.qid = pc.qid
JOIN wikidata_location_p31 p ON p.qid = pc.qid
GROUP BY l.qid, l.kind, pc.p31_count
ORDER BY pc.p31_count DESC
LIMIT {SAMPLE_LIMIT};
""")
show("Sample: locations with many P31 classes", multi_p31_samples)


## 7. Coordinate coverage and coordinate sanity

In [ ]:

coord_summary = q("""
SELECT
    kind,
    COUNT(*) AS locations,
    SUM(CASE WHEN lat IS NOT NULL AND lon IS NOT NULL THEN 1 ELSE 0 END) AS with_coords,
    SUM(CASE WHEN (lat IS NULL) <> (lon IS NULL) THEN 1 ELSE 0 END) AS partial_coords,
    SUM(CASE WHEN lat < -90 OR lat > 90 OR lon < -180 OR lon > 180 THEN 1 ELSE 0 END) AS invalid_range,
    SUM(CASE WHEN lat = 0 AND lon = 0 THEN 1 ELSE 0 END) AS zero_zero
FROM wikidata_location
GROUP BY kind
ORDER BY locations DESC;
""")
coord_summary["with_coords_pct"] = coord_summary["with_coords"] / coord_summary["locations"] * 100
show("Coordinate coverage by kind", coord_summary)

invalid_coord_samples = q(f"""
SELECT qid, kind, lat, lon
FROM wikidata_location
WHERE (lat IS NULL) <> (lon IS NULL)
   OR lat < -90 OR lat > 90 OR lon < -180 OR lon > 180
   OR (lat = 0 AND lon = 0)
ORDER BY kind, qid
LIMIT {SAMPLE_LIMIT};
""")
show("Sample: suspicious coordinates", invalid_coord_samples)


In [ ]:

coord_piles = q(f"""
SELECT
    ROUND(lat, 4) AS lat4,
    ROUND(lon, 4) AS lon4,
    COUNT(*) AS locations,
    COUNT(DISTINCT kind) AS kinds,
    GROUP_CONCAT(DISTINCT kind) AS kind_list
FROM wikidata_location
WHERE lat IS NOT NULL AND lon IS NOT NULL
GROUP BY ROUND(lat, 4), ROUND(lon, 4)
HAVING COUNT(*) >= 10
ORDER BY locations DESC
LIMIT {TOP_LIMIT};
""")
show("Suspicious coordinate piles rounded to 4 decimals", coord_piles)


## 8. Name string and normalization quality


In [ ]:

with get_conn() as conn:
    conn.create_function("nte_norm_qc", 1, normalize_name_for_qc)
    conn.create_function("has_control_chars", 1, has_control_chars)
    name_quality = pd.read_sql_query("""
    WITH metrics AS (
        SELECT 'empty name_norm' AS check_name, COUNT(*) AS rows
        FROM wikidata_location_name
        WHERE name_norm = ''

        UNION ALL
        SELECT 'leading/trailing whitespace in raw name', COUNT(*)
        FROM wikidata_location_name
        WHERE name <> TRIM(name)

        UNION ALL
        SELECT 'raw name contains newline/tab/control char', COUNT(*)
        FROM wikidata_location_name
        WHERE has_control_chars(name) = 1

        UNION ALL
        SELECT 'raw name contains replacement character', COUNT(*)
        FROM wikidata_location_name
        WHERE name LIKE '%�%'

        UNION ALL
        SELECT 'very long raw name (>120 chars)', COUNT(*)
        FROM wikidata_location_name
        WHERE LENGTH(name) > 120

        UNION ALL
        SELECT 'HTML/XML-looking raw name', COUNT(*)
        FROM wikidata_location_name
        WHERE name LIKE '%<%' AND name LIKE '%>%'

        UNION ALL
        SELECT 'name_norm differs from QC normalizer', COUNT(*)
        FROM wikidata_location_name
        WHERE name_norm <> nte_norm_qc(name)

        UNION ALL
        SELECT 'labels with parenthetical disambiguator', COUNT(*)
        FROM wikidata_location_name
        WHERE name LIKE '%(%)%'

        UNION ALL
        SELECT 'labels with comma qualifier', COUNT(*)
        FROM wikidata_location_name
        WHERE name LIKE '%,%'
    )
    SELECT * FROM metrics ORDER BY rows DESC;
    """, conn)
show("Name quality flags", name_quality)


In [ ]:

with get_conn() as conn:
    conn.create_function("nte_norm_qc", 1, normalize_name_for_qc)
    conn.create_function("has_control_chars", 1, has_control_chars)
    suspicious_name_samples = pd.read_sql_query(f"""
    SELECT qid, wd_lang, geo_lang, term_type, name, name_norm,
           nte_norm_qc(name) AS qc_norm,
           LENGTH(name) AS name_len
    FROM wikidata_location_name
    WHERE name_norm = ''
       OR name <> TRIM(name)
       OR has_control_chars(name) = 1
       OR name LIKE '%�%'
       OR LENGTH(name) > 120
       OR (name LIKE '%<%' AND name LIKE '%>%')
       OR name_norm <> nte_norm_qc(name)
    ORDER BY name_len DESC, qid
    LIMIT {SAMPLE_LIMIT};
    """, conn)
show("Sample: suspicious names", suspicious_name_samples)


In [ ]:

within_entity_duplicates = q(f"""
SELECT
    qid,
    geo_lang,
    name_norm,
    COUNT(*) AS rows,
    COUNT(DISTINCT wd_lang) AS wd_langs,
    COUNT(DISTINCT name) AS raw_names,
    GROUP_CONCAT(DISTINCT wd_lang) AS wd_lang_list,
    GROUP_CONCAT(DISTINCT name) AS raw_name_list
FROM wikidata_location_name
WHERE geo_lang IS NOT NULL
GROUP BY qid, geo_lang, name_norm
HAVING COUNT(*) > 1
ORDER BY rows DESC, wd_langs DESC
LIMIT {SAMPLE_LIMIT};
""")
show("Within-entity duplicates after geo_lang/name_norm collapse", within_entity_duplicates)

lookup_ambiguity_top = q(f"""
SELECT
    geo_lang,
    name_norm,
    COUNT(DISTINCT qid) AS qids,
    COUNT(*) AS rows,
    COUNT(DISTINCT term_type) AS term_types,
    COUNT(DISTINCT kind) AS kinds,
    GROUP_CONCAT(DISTINCT kind) AS kind_list
FROM (
    SELECT n.geo_lang, n.name_norm, n.qid, n.term_type, l.kind
    FROM wikidata_location_name n
    JOIN wikidata_location l ON l.qid = n.qid
    WHERE n.geo_lang IS NOT NULL
)
GROUP BY geo_lang, name_norm
HAVING COUNT(DISTINCT qid) > 1
ORDER BY qids DESC, rows DESC
LIMIT {TOP_LIMIT};
""")
show("Most ambiguous lookup keys: geo_lang + name_norm → multiple QIDs", lookup_ambiguity_top)


In [ ]:

ambiguity_by_lang = q(f"""
WITH keys AS (
    SELECT geo_lang, name_norm, COUNT(DISTINCT qid) AS qids
    FROM wikidata_location_name
    WHERE geo_lang IS NOT NULL
    GROUP BY geo_lang, name_norm
), per_lang AS (
    SELECT
        geo_lang,
        COUNT(*) AS lookup_keys,
        SUM(CASE WHEN qids > 1 THEN 1 ELSE 0 END) AS ambiguous_keys,
        SUM(CASE WHEN qids > 1 THEN qids ELSE 0 END) AS ambiguous_qid_assignments,
        MAX(qids) AS max_qids_for_one_key
    FROM keys
    GROUP BY geo_lang
)
SELECT
    geo_lang,
    lookup_keys,
    ambiguous_keys,
    ROUND(100.0 * ambiguous_keys / lookup_keys, 3) AS ambiguous_keys_pct,
    ambiguous_qid_assignments,
    max_qids_for_one_key
FROM per_lang
ORDER BY ambiguous_keys DESC
LIMIT {TOP_LIMIT};
""")
show("Lookup ambiguity by language bucket", ambiguity_by_lang)


## 9. Truncation bias checks


In [ ]:

qid_summary = q(f"""
WITH q AS (
    SELECT {qid_num_expr('qid')} AS qid_num, kind
    FROM wikidata_location
)
SELECT
    COUNT(*) AS valid_qid_rows,
    MIN(qid_num) AS min_qid_num,
    MAX(qid_num) AS max_qid_num,
    AVG(qid_num) AS avg_qid_num
FROM q
WHERE qid_num IS NOT NULL;
""")
show("QID numeric summary", qid_summary.T.reset_index().rename(columns={"index": "metric", 0: "value"}))

qid_buckets = q(f"""
WITH q AS (
    SELECT {qid_num_expr('qid')} AS qid_num, kind
    FROM wikidata_location
    WHERE {qid_num_expr('qid')} IS NOT NULL
), bucketed AS (
    SELECT
        (qid_num / 10000000) * 10000000 AS qid_bucket_start,
        kind,
        COUNT(*) AS locations
    FROM q
    GROUP BY qid_bucket_start, kind
)
SELECT * FROM bucketed ORDER BY qid_bucket_start, kind;
""")
show("QID bucket × kind distribution", qid_buckets)


In [ ]:

qid_total_by_bucket = qid_buckets.groupby("qid_bucket_start", as_index=False)["locations"].sum()
ax = qid_total_by_bucket.plot.bar(x="qid_bucket_start", y="locations", legend=False, figsize=(12, 5))
ax.set_title("Locations by QID bucket (10M-wide buckets)")
ax.set_xlabel("QID numeric bucket start")
ax.set_ylabel("locations")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:

qid_ventiles = q(f"""
WITH q AS (
    SELECT {qid_num_expr('qid')} AS qid_num, kind
    FROM wikidata_location
    WHERE {qid_num_expr('qid')} IS NOT NULL
), ranked AS (
    SELECT qid_num, kind, NTILE(20) OVER (ORDER BY qid_num) AS qid_ventile
    FROM q
)
SELECT
    qid_ventile,
    MIN(qid_num) AS min_qid_num,
    MAX(qid_num) AS max_qid_num,
    COUNT(*) AS locations,
    COUNT(DISTINCT kind) AS kind_count
FROM ranked
GROUP BY qid_ventile
ORDER BY qid_ventile;
""")
show("QID ventiles", qid_ventiles)

highest_qid_samples = q(f"""
SELECT qid, kind, lat, lon
FROM wikidata_location
WHERE {qid_num_expr('qid')} IS NOT NULL
ORDER BY {qid_num_expr('qid')} DESC
LIMIT {SAMPLE_LIMIT};
""")
show("Sample: highest-QID surviving locations", highest_qid_samples)


In [ ]:

qid_kind_pivot = qid_buckets.pivot_table(
    index="qid_bucket_start",
    columns="kind",
    values="locations",
    aggfunc="sum",
    fill_value=0,
)
# Show only the most common kinds as columns to keep the table readable.
top_kind_cols = kind_overview.head(15)["kind"].tolist()
qid_kind_pivot_top = qid_kind_pivot[[c for c in top_kind_cols if c in qid_kind_pivot.columns]]
show("QID bucket pivot for top kinds", qid_kind_pivot_top.reset_index())


## 10. Actionable issue scorecard

This scorecard distills the preceding sections into a few counts that are likely to affect the lookup engine.


In [ ]:

issue_queries = [
    ("locations with no names", "SELECT COUNT(*) FROM wikidata_location l LEFT JOIN wikidata_location_name n ON n.qid = l.qid WHERE n.qid IS NULL"),
    ("locations with no mapped-language names", "SELECT COUNT(*) FROM wikidata_location l LEFT JOIN wikidata_location_name n ON n.qid = l.qid AND n.geo_lang IS NOT NULL WHERE n.qid IS NULL"),
    ("name rows with geo_lang NULL", "SELECT COUNT(*) FROM wikidata_location_name WHERE geo_lang IS NULL"),
    ("name rows with empty name_norm", "SELECT COUNT(*) FROM wikidata_location_name WHERE name_norm = ''"),
    ("locations with invalid/partial coordinates", "SELECT COUNT(*) FROM wikidata_location WHERE (lat IS NULL) <> (lon IS NULL) OR lat < -90 OR lat > 90 OR lon < -180 OR lon > 180"),
    ("locations with no P31 row", "SELECT COUNT(*) FROM wikidata_location l LEFT JOIN wikidata_location_p31 p ON p.qid = l.qid WHERE p.qid IS NULL"),
    ("locations with GeoNames link", "SELECT COUNT(DISTINCT qid) FROM wikidata_location_geonames"),
    ("ambiguous mapped lookup keys", "SELECT COUNT(*) FROM (SELECT geo_lang, name_norm FROM wikidata_location_name WHERE geo_lang IS NOT NULL GROUP BY geo_lang, name_norm HAVING COUNT(DISTINCT qid) > 1)"),
]

total_locations = scalar("SELECT COUNT(*) FROM wikidata_location")
total_names = scalar("SELECT COUNT(*) FROM wikidata_location_name")
rows = []
for label, sql in issue_queries:
    n = scalar(sql)
    denominator = total_names if label.startswith("name rows") else total_locations
    rows.append({
        "metric": label,
        "n": n,
        "denominator": denominator,
        "pct_of_denominator": pct(n, denominator),
    })
issue_scorecard = pd.DataFrame(rows).sort_values("n", ascending=False)
show("QC issue scorecard", issue_scorecard)
